<a href="https://www.kaggle.com/code/mariopaerle/modern-llms-model-with-vathos?scriptVersionId=300812391" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, IterableDataset
from datasets import load_dataset
from transformers import AutoTokenizer

class FineWebStreamingDataset(IterableDataset):
    def __init__(self, hf_dataset, tokenizer, max_length=512):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __iter__(self):
        for item in self.dataset:
            text = item.get('text', '')
            if not text or len(text) < 10: 
                continue
            
            tokenized = self.tokenizer(
                text,
                max_length=self.max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            
            yield {
                'input_ids': tokenized['input_ids'].squeeze(0),
                'attention_mask': tokenized['attention_mask'].squeeze(0)
            }

def get_dataloaders(model_name="gpt2", batch_size_train=6, batch_size_val=4, max_length=512, val_size=1000):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    ds = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True)
    #ds = load_dataset("mlabonne/guanaco-llama2-1k", split="train", streaming=True)    

    shuffled_ds = ds.shuffle(seed=42, buffer_size=10000)
    
    val_raw = shuffled_ds.take(val_size)
    train_raw = shuffled_ds.skip(val_size)
    
    train_dataset = FineWebStreamingDataset(train_raw, tokenizer, max_length=max_length)
    val_dataset = FineWebStreamingDataset(val_raw, tokenizer, max_length=max_length)

    train_loader = DataLoader(train_dataset, batch_size=batch_size_train)
    val_loader = DataLoader(val_dataset, batch_size=batch_size_val)
    
    return train_loader, val_loader, tokenizer


In [15]:
VAL_SIZE = 0
train_loader, val_loader, tokenizer = get_dataloaders(
    model_name="gpt2", 
    batch_size_train=15, 
    batch_size_val=4, 
    max_length=256,
    val_size=VAL_SIZE
)

TOKEN_PAD = tokenizer.pad_token_id
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Padding ID: {TOKEN_PAD}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Tokenizer vocab size: 50257
Padding ID: 50256


In [2]:
!pip uninstall aplos -y
!pip install -q git+https://github.com/MarioPaerle/Aplos@VathosUI

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 68.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 83.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.


In [5]:
from Vathos.blocks import *


def soft_cap_logits(logits, cap=10.0):
    return F.hardtanh(logits, min_val=-cap, max_val=cap)

def trapezoidal_lr(step,
                   warmup_steps,
                   plateau_steps,
                   total_steps):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    elif step < warmup_steps + plateau_steps:
        return 1.0
    else:
        decay_steps = total_steps - warmup_steps - plateau_steps
        step_in_decay = step - warmup_steps - plateau_steps
        return max(0.0, 1.0 - step_in_decay / max(1, decay_steps))

In [6]:
!pip install git+https://github.com/KellerJordan/Muon.git

  Cloning https://github.com/KellerJordan/Muon.git to /tmp/pip-req-build-apxn4g06
  Running command git clone --filter=blob:none --quiet https://github.com/KellerJordan/Muon.git /tmp/pip-req-build-apxn4g06
  Resolved https://github.com/KellerJordan/Muon.git to commit 6399c658d3c4a3356ba823fa6664b10e23871068
  Preparing metadata (setup.py) ... done
  Created wheel for muon-optimizer: filename=muon_optimizer-0.1.0-py3-none-any.whl size=7141 sha256=17d301c71271ee551aeeb75542220e2fd0c878ecb97953d633a073a05a50a04d
  Stored in directory: /tmp/pip-ephem-wheel-cache-b5360a1z/wheels/b9/0f/69/c35113e898da3fd9c4218a4fbd45bb54a669e69b9e52c6cb20
Successfully built muon-optimizer


In [7]:
from Vathos.optimizers import *

In [103]:
device = 'cuda'
model = SequenceModel(
        vocab_size=50304,
        d_model=384,
        n_layers=10,
        max_len=256,
        pos_encoder=False,
        rope=True,
        embedder=EasyEmbedder,
        channel_mixer=UDLPReLU2,
        channel_args={'expand': 4},
        spatial_mixer=GQANOV,
        spatial_args={'n_heads': 8, 'causal': True, 'n_kv_heads': 2},
        # spatial_args={'n_heads': 8, 'causal': True},
        weight_tying=True,
        norm=RMSNorm,
        baseblock=Block1d,
        unet_skips=False
    ).to(device)

model.summary()
model.autosave = True
model.autosave_overwrite = True
# model.load_checkpoint("/kaggle/working/-checkpoint.pt")
# model.name = 'PiCO2_alpha'

# model = torch.compile(model)

||FLAG LV.1|| rope=True works only with MultiheadAttentionMixer, which you seem not to be using right?
VATHOS  Summary:
SequenceModel(d_model=384, n_layer=10)
	 - VOCAB_SIZE:: 50304
	 - D_MODEL:: 384
	 - N_LAYERS:: 10

	 - Embedder: EasyEmbedder - O(L d)
	 - Unembedder: UnbiasedLinear - O(L d^2)
	 - Spatial Mixer: GroupedQueryAttentionNOV({'n_heads': 8, 'causal': True, 'n_kv_heads': 2}) - O(L d)
	 - Channel Mixer: UDLPReLU2({'expand': 4}) - O(L d)
Num Parameters: 34_439_424
Num Trainable Parameters: 34_439_424
Total Complexity: O(L d^2)


In [105]:
from muon import MuonWithAuxAdam
from torch.optim import AdamW

hidden_weights = []
hidden_gains_biases = []
nonhidden_params = [] 

embed_and_head_keywords = ['embedding', 'unembedder', 'embedder', 'unembed' 'output']

for name, p in model.named_parameters():
    if any(keyword in name for keyword in embed_and_head_keywords):
        print(name)
        nonhidden_params.append(p)
    elif p.ndim >= 2:
        hidden_weights.append(p)
    else:
        hidden_gains_biases.append(p)

param_groups = [
    dict(
        params = hidden_weights,
        use_muon = True,
        lr = 1e-2,        
        weight_decay = 0.01
    ),
    dict(
        params = hidden_gains_biases + nonhidden_params,
        use_muon = False,
        lr = 5e-3,          
        betas = (0.9, 0.999), 
        weight_decay = 0.01
    ),
]

optimizer = MuonWithAuxAdam(param_groups)
# optimizer = HybridMuonWithAuxAdam(param_groups)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: trapezoidal_lr(
        step,
        warmup_steps=1_000,
        plateau_steps=15_000, # 10_000
        total_steps=32_000 # 15000
    )
)  

embedder.embedding.weight


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_scheduler(scheduler_fn, total_steps, title="Learning Rate Schedule"):
    steps = np.arange(total_steps)
    lrs = [scheduler_fn(step) for step in steps]
    
    plt.figure(figsize=(10, 5))
    plt.plot(steps, lrs, linewidth=2)
    plt.xlabel("Steps")
    plt.ylabel("LR Multiplier")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()

# Usage:
plot_scheduler(
    lambda step: trapezoidal_lr(step, warmup_steps=1_000, plateau_steps=15_000, total_steps=32_000),
    total_steps=32_000
)

In [17]:
import torch.distributed as dist

dist.init_process_group(
    backend='nccl',
    init_method='tcp://127.0.0.1:29500',
    world_size=1,
    rank=0
)

In [ ]:
from tqdm.notebook import tqdm
import torch
import torch.nn.functional as F
from timeit import default_timer as timer

set_vathos_mode('production')
scaler = torch.amp.GradScaler("cuda") 

model.train()
SPE = 1000 
num_epochs = 20
acc_step = 1
finished = False
seq_len = 32
step = 0

train_iter = iter(train_loader)
optimizer.zero_grad(set_to_none=True)

global_batch_step = 0
loss_history = []
start_time = timer()
print("Timer Started")

for epoch in range(num_epochs):
    total_loss = 0
    pbar = tqdm(range(SPE), desc=f"Epoch {epoch+1}/{num_epochs}")
      
    if finished:
        break
        
    for count in pbar:
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        input_ids = batch['input_ids'].to(device)
        
        x = input_ids[:, :seq_len-1].contiguous()
        y = input_ids[:, 1:seq_len].contiguous()

        
        with torch.amp.autocast("cuda"):
            pred = model(x)
            B, T, V = pred.shape
            loss = F.cross_entropy(
                pred.view(-1, V), 
                y.view(-1), 
                ignore_index=tokenizer.pad_token_id
            )
            loss = loss / acc_step 

        scaler.scale(loss).backward()
        
        global_batch_step += 1
        
        if global_batch_step % acc_step == 0:
            
            # scaler.unscale_(optimizer)
            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            
            old_scale = scaler.get_scale()
            scaler.update()
            new_scale = scaler.get_scale()
            
            if new_scale >= old_scale:
                scheduler.step()
                
            optimizer.zero_grad(set_to_none=True)
        
        if step % 500 == 0:
            seq_len = min(seq_len*2, 256)
            acc_step = min(acc_step + 1, 5)
        step += 1

        ########################################################################### NON MODIFICABILE
        current_lr = optimizer.param_groups[0]['lr']
        loss_val = loss.item() * acc_step
        total_loss += loss_val
        

        k = 20
        loss_history.append(loss_val)
        if len(loss_history) > k:
            loss_history.pop(0)
            
        n = len(loss_history)
        weights = [i + 1 for i in range(n)]
        weighted_loss = sum(w * l for w, l in zip(weights, loss_history)) / sum(weights)

        """if weighted_loss <= 5.0:
            print(f"FINITOOOO in {timer() - start_time:.2f}s and {count + epoch*SPE} steps")
            finished = True
            break"""

        pbar.set_postfix({
            'avg_loss': f"{total_loss/(count+1):.4f}", 
            'lr': f"{current_lr:.2e}",
            'MA@20': weighted_loss
        })

        if hasattr(model, 'module'):
            model.module.register_loss(loss_val)
        else:
            model.register_loss(loss_val)
            
    if not finished and (global_batch_step % acc_step != 0):
        scaler.step(optimizer)
        old_scale = scaler.get_scale()
        scaler.update()
        if scaler.get_scale() >= old_scale:
            scheduler.step()
        optimizer.zero_grad(set_to_none=True)
    
    if hasattr(model, 'module'):
        model.module.register_epoch()
    else:
        model.register_epoch()
    print(f"Elapsed time: {timer() - start_time}s")

Vathos: Switched to PRODUCTION mode (Ready for torch.compile, Profiling Disabled).
Timer Started


Epoch 1/20:   0%|          | 0/1000 [00:00<?, ?it/s]

Elapsed time: 180.5881674859993s


Epoch 2/20:   0%|          | 0/1000 [00:00<?, ?it/s]

Elapsed time: 443.7701514590008s


...

Epoch 13/20:   0%|          | 0/1000 [00:00<?, ?it/s]

Elapsed time: 3337.977075560999s


Epoch 14/20:   0%|          | 0/1000 [00:00<?, ?it/s]

Elapsed time: 3603.2504694420004s


Epoch 15/20:   0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
acc_step

In [ ]:
print(scheduler.last_epoch)

In [ ]:
from Vathos.functions import plot

plot(model._losses_per_epoch)
print(model.steps)
print(min(model._losses_per_epoch))

In [47]:
import gc; gc.collect()
torch.cuda.empty_cache()
# del model
# del optimizer


In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0, repetition_penalty=1.2, top_k=None, device='cuda'):
    model.eval()
    
    input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)  # (1, T)
    
    for _ in range(max_new_tokens):
        input_ids_cond = input_ids[:, -model.max_len:]
        
        logits = model(input_ids_cond)
        logits = logits[:, -1, :]     
        logits = logits / temperature
        if repetition_penalty != 1.0:
            for token_id in set(input_ids[0].tolist()):
                logits[0, token_id] /= repetition_penalty
        
        if top_k is not None:
            top_values, top_indices = torch.topk(logits, top_k)
            probs = torch.zeros_like(logits).scatter_(1, top_indices, F.softmax(top_values, dim=-1))
        else:
            probs = F.softmax(logits, dim=-1)
        
        next_token = torch.multinomial(probs, num_samples=1)  # (1,1)
        
        input_ids = torch.cat([input_ids, next_token], dim=1)
    
    output_text = tokenizer.decode(input_ids.squeeze().tolist())
    return output_text

prompt = "DNA is a molecule made of"
generated_text = generate(model, tokenizer, prompt, max_new_tokens=100, temperature=0.5, top_k=40, repetition_penalty=1.1, device=device)
print(generated_text)

In [ ]:
tokenizer

In [102]:
prompt = "Hemoglobin"
input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
out = model.generate(input_ids, 100, temperature=0.1, custom_generate=False, repetition_penalty=1.2, top_k=20)
print(tokenizer.decode(out.squeeze().tolist()))



Simple Gen:   0%|          | 0/100 [00:00<?, ?it/s]

Simple Gen:   1%|          | 1/100 [00:00<01:03,  1.57it/s]

Simple Gen:  15%|█▌        | 15/100 [00:00<00:03, 26.84it/s]

Simple Gen:  28%|██▊       | 28/100 [00:00<00:01, 47.52it/s]

Simple Gen:  39%|███▉      | 39/100 [00:00<00:00, 61.29it/s]

Simple Gen:  50%|█████     | 50/100 [00:01<00:00, 70.90it/s]

Simple Gen:  61%|██████    | 61/100 [00:01<00:00, 78.06it/s]

Simple Gen:  71%|███████   | 71/100 [00:01<00:00, 81.60it/s]

Simple Gen:  81%|████████  | 81/100 [00:01<00:00, 83.34it/s]

Simple Gen: 100%|██████████| 100/100 [00:01<00:00, 61.40it/s][A

Hemoglobin is a type of liver transplant, or other body. It can be found in the kidneys and kidneys.
The kidneys are known to cause kidney failure, but it may also appear that they have been linked to kidney failure. The kidneys is responsible for kidney failure, which is called kidney failure.
A blood clotting fluid is a form of kidney failure. This is an essential part of kidney failure. In this case you will need to repair kidney failure if your kidneys is damaged.
Blood pressure
